In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Define directories
DATASET_DIR = Path('/content/drive/MyDrive/la/la')
print("Dataset path:", DATASET_DIR)
print("Dataset folder exists:", DATASET_DIR.exists())

SEED = 42
np.random.seed(SEED)

# ==========================================
# TASK 1: PURE NUMPY LINEAR ALGEBRA
# ==========================================
print("\n--- Starting Task 1: Linear Algebra Methods ---")

# Task 1.1: Geometric Transformations
A = np.array([[4, 2], [2, 3]], dtype=float)
eigenvalues, eigenvectors = np.linalg.eig(A)
print("Eigenvalues:", eigenvalues)
print("Eigenvectors:\n", eigenvectors)

# Verify Av = lambda v
def verify_eigen(A, evals, evecs):
    for i in range(len(evals)):
        v = evecs[:, i]
        lam = evals[i]
        lhs = np.dot(A, v)
        rhs = lam * v
        assert np.allclose(lhs, rhs), f"Verification failed for index {i}"
    print("Eigenvalue identity Av = lambda v verified successfully!")

verify_eigen(A, eigenvalues, eigenvectors)

# Task 1.2: Gram-Schmidt QR Decomposition
def custom_qr(M):
    """Custom QR decomposition for a 3x3 matrix using Gram-Schmidt."""
    m, n = M.shape
    Q = np.zeros((m, n))
    for j in range(n):
        v = M[:, j]
        for i in range(j):
            q = Q[:, i]
            v = v - np.dot(q, M[:, j]) * q
        norm_v = np.linalg.norm(v)
        Q[:, j] = v / norm_v
    R = np.dot(Q.T, M)
    return Q, R

# Test custom QR with a random 3x3 matrix
np.random.seed(SEED)
M_test = np.random.rand(3, 3)
Q_custom, R_custom = custom_qr(M_test)
print("Custom QR Q^T * Q (Identity check):\n", np.round(np.dot(Q_custom.T, Q_custom), 4))

# Task 1.3: Iterative QR Algorithm for Eigenvalues
Ak = A.copy()
for _ in range(50):
    Q_k, R_k = custom_qr(Ak)
    Ak = np.dot(R_k, Q_k)

print("Eigenvalues via Iterative QR Algorithm (Diagonal of A_50):", np.diagonal(Ak))


# ==========================================
# TASK 2: EIGENFACES CLASSIFICATION USING WRAPPER
# ==========================================
print("\n--- Starting Task 2: Eigenfaces & Classification ---")

# Simulating importing/initializing the wrapper class as discussed in the tutorial
face_app = FaceRecognizerWrapper(img_size=300, n_components=50, seed=SEED)

# Load dataset and preprocess (Task 2.1)
face_app.load_and_preprocess_data(DATASET_DIR)

# Split dataset (80% train / 20% test)
face_app.split_data(test_size=0.20)

# Apply PCA (Eigenfaces) and Train Classifier (Task 2.2 & 2.3)
face_app.apply_pca_and_train()

# Evaluation & Metrics
accuracy, report, conf_matrix, y_pred = face_app.evaluate()
print(f"\nTest Accuracy: {round(accuracy * 100, 2)}%")
print("\nClassification Report:\n", report)

# Plot Confusion Matrix
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=face_app.class_names)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix for Face Classification")
plt.show()